# ADAPT-VQE vs the classical methods — Li₂S at 24 qubits, 2.10 Å

One geometry, one question: **how close does a strong conventional VQE get, and what
does it cost, against the classical methods and against HI-VQE?**

This notebook does *not* compute the exact energy — we already have that. At
CAS(12e,12o) the exact answer is CASCI, a direct diagonalization of 853,776
determinants, and it is the number everything here is scored against.

### Why ADAPT-VQE and not plain VQE

Comparing against a hand-picked hardware-efficient circuit optimized by SPSA would be
a straw man. Everyone already knows that fails at 24 qubits — 322 parameters, barren
plateaus, and the immediate objection *"you chose a bad circuit."*

[ADAPT-VQE](https://www.nature.com/articles/s41467-019-10988-2) removes that
objection by **building its own ansatz**. Each round it measures the energy gradient
of every operator in a pool, appends the single best one, re-optimizes every
parameter, and repeats until nothing left in the pool is worth adding. The circuit is
a result, not a choice.

### The comparison is deliberately unfair — in VQE's favour

The VQE here is given a best case no real device could ever have:

| | This notebook | A real VQE |
|---|---|---|
| Expectation values | **Exact** | Estimated from shots |
| Shots | **Infinite** | Finite, noisy |
| Measurement cost | **Zero** | 15,697 Pauli words per step |
| Hardware noise | **None** | Substantial |
| Ansatz | **Built by the algorithm** | Chosen up front |

So any gap left against CASCI is the **ansatz expressiveness ceiling** — the one thing
better hardware does not fix. If it misses chemical accuracy here, that conclusion is
airtight. The converse does *not* hold: success here would not mean a real VQE
succeeds, because the 15,697 measurement settings per iteration are still owed.

### What you get at the end

A single table putting the classical methods and the quantum methods on the same
footing, at the same geometry, in the same active space:

| Kind | Method | What it is |
|---|---|---|
| Classical | RHF | mean field, the starting point |
| Classical | MP2 / CCSD / CCSD(T) | the standard correlated hierarchy |
| Classical | **CASCI** | **exact in this active space — the yardstick** |
| Quantum | ADAPT-VQE | quantum computes the energy |
| Quantum | HI-VQE | quantum proposes, classical solves |

**Runtime: roughly 1–2 hours** on a free Colab CPU. Sections 1–4 take minutes; §5 is
the long one. Nothing here needs a GPU or a QPU.

## 1. Install

Same stack as the main notebook. `openfermion` is not needed here at all — nothing in
the ADAPT-VQE path touches a Pauli decomposition, which is rather the point.

In [ ]:
!pip install -q "pyscf>=2.5" "qiskit>=1.2,<3" "qiskit-aer>=0.15" "matplotlib>=3.8"

In [ ]:
import importlib
from importlib import metadata

for module in ("numpy", "scipy", "pyscf", "qiskit", "qiskit_aer", "matplotlib"):
    try:
        importlib.import_module(module)
        name = {"qiskit_aer": "qiskit-aer"}.get(module, module)
        print(f"  ok        {module:<14} {metadata.version(name)}")
    except Exception as error:
        print(f"  PROBLEM   {module:<14} {type(error).__name__}: {error}")


## 2. Get the workflow

`bhargav2603/qubit_run` is public, so this is a plain anonymous clone.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

OWNER, NAME, BRANCH, FOLDER = "bhargav2603", "qubit_run", "main", "li2s_24"
URL = f"https://github.com/{OWNER}/{NAME}.git"
target = Path("/content/repo")

if target.exists():
    shutil.rmtree(target)
finished = subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", BRANCH, URL, str(target)],
    capture_output=True, text=True,
)
if finished.returncode != 0:
    raise SystemExit(f"Clone failed: {finished.stderr.strip().splitlines()[-1]}")

candidates = sorted(target.rglob("run.py"))
matching = [p for p in candidates if p.parent.name == FOLDER] or candidates
ROOT = matching[0].parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

commit = subprocess.run(
    ["git", "-C", str(target), "log", "-1", "--format=%h %ad %s", "--date=short"],
    capture_output=True, text=True,
).stdout.strip()
print(f"Working in : {ROOT}")
print(f"Commit     : {commit}")
print("adapt_vqe.py present:", (ROOT / "adapt_vqe.py").is_file())


## 3. Prove the ADAPT-VQE code before spending an hour on it

Thirteen checks, none of them against a recorded number — every one is against an
independent computation:

* the generator `A = T - T†` really is antisymmetric, without which `exp(A)` is not
  orthogonal and the state silently stops being normalized;
* the scaled-Taylor exponential against **`scipy.linalg.expm`** on the dense generator;
* the analytic adjoint gradients against **central finite differences** — this is the
  one that matters most, because a wrong gradient does not crash, it just converges
  somewhere plausible and wrong;
* the pool-screening gradient against finite differences too;
* and on a small system where the answer is known exactly: ADAPT-VQE reaches it,
  never dips below it, and decreases monotonically.

The self-test at the end is the main workflow's, unchanged. **If either is not green,
stop.**

In [ ]:
!python -m unittest tests.test_adapt_vqe -v 2>&1 | tail -20
!python run.py selftest 2>&1 | tail -3

## 4. Build the Hamiltonian at 2.10 Å and prove it correct

Same cache, same receipt, same gate as the main workflow — ADAPT-VQE reads exactly
what HI-VQE reads, so the comparison is against an identical Hamiltonian rather than
a re-derived one.

2.10 Å is this folder's own CASCI/STO-3G minimum. It is also the geometry where the
reference is known to be sound: the singlet clearly dominates, so there is no
singlet–triplet near-degeneracy for the reference solver to trip over.

`classical` then runs MP2, CCSD and CCSD(T) frozen to the identical active space.
Watch the **T1 diagnostic** — below ~0.02 the single-reference picture holds, which is
what makes CCSD(T) a fair gold standard *here* and not at a stretched bond.

In [ ]:
!python run.py prepare   --distance 2.10
!python run.py validate  --distance 2.10
!python run.py classical --distance 2.10

## 5. ADAPT-VQE — the long cell

Each round: screen the whole pool for the largest energy gradient, append that one
operator, re-optimize every parameter with L-BFGS, repeat.

Two implementation choices make this affordable, and both are worth knowing because
they are what separates a 1-hour run from a 10-hour one:

* **The entire pool is screened for the price of one sigma product.** For an
  anti-Hermitian generator, `⟨ψ|[H,A]|ψ⟩ = 2⟨Hψ, Aψ⟩` — so `Hψ` is computed once
  (~6 s over 853,776 determinants) and every one of the ~700 operators then costs a
  sparse multiply and a dot product.
* **Optimizer gradients are analytic, via an adjoint sweep.** A finite difference over
  *k* parameters needs *k+1* sigma products per gradient. The backward sweep gets all
  *k* from **one**.

Read three things in the output:

* **`err (mHa)`** — the running error against CASCI. The 1.6 mHa pass mark is the bar.
* **`k`** — ansatz length. Each operator is one more parameter, and on real hardware
  one more block of circuit depth.
* **the stopping line** — `converged` on the gradient criterion means the pool had
  nothing left worth adding, and *that* is the ansatz ceiling. Stopping at the
  operator cap instead means it was still improving and simply ran out of budget —
  a different, weaker statement.

**Measured cost, so you can budget it.** Each round is one pool screen (~45 s over 702
operators) plus an L-BFGS re-optimization that needs only ~6-7 energy evaluations
thanks to the warm start - about **2-3 minutes per operator, growing with ansatz
length**. The cell below asks for 30, which is ~1.5 hours.

**It checkpoints after every operator.** The result JSON is rewritten each round with
`"partial": true`, so a Colab disconnect costs you the remaining operators and nothing
else - sections 7 and 8 read a partial file perfectly well.

Raise `--max-operators` if the error is still falling at the last row: that means it
ran out of budget rather than converging, and the number is not yet the method's best.

In [ ]:
!python run.py adapt --distance 2.10 --max-operators 30

### Optional: the generalized pool

The default pool is the UCCSD one — excitations out of the orbitals Hartree–Fock
occupies and into the ones it does not. `--generalized-pool` drops that distinction
and allows every orbital pair: a much larger pool, a strictly more expressive ansatz,
and a slower run.

At 2.10 Å it should change little, because the reference determinant genuinely does
dominate here. It is the knob that matters at a *stretched* bond, where "occupied" and
"virtual" stop being meaningful labels. Skip this cell on a first pass.

In [ ]:
# Optional and slower. Uncomment to run.
# !python run.py adapt --distance 2.10 --max-operators 30 --generalized-pool

## 6. HI-VQE at the same geometry, for the comparison

The same cache, the same 24 qubits, the same 853,776-determinant space. The only thing
that changes is **where the energy comes from**: ADAPT-VQE takes the expectation value
of an optimized quantum state; HI-VQE samples the circuit for electron configurations
and diagonalizes **H** exactly inside the subspace they span.

Minutes, not hours — which is itself part of the result.

In [ ]:
!python run.py hivqe --distance 2.10 --max-determinants 20000 --max-iterations 12

## 7. The comparison — classical vs quantum, one table

Everything at 2.10 Å, in the same active space, scored against CASCI.

Two columns deserve attention beyond the error:

* **correlation recovered** — of the energy CASCI finds below Hartree–Fock, what
  fraction did this method capture? A method can look respectable in absolute
  hartree and still be missing a third of the correlation energy.
* **cost** — the units are not comparable across methods and are shown anyway,
  because "how accurate" is meaningless without "for what."

In [ ]:
import json
from pathlib import Path

HARTREE_TO_MHA = 1000.0
results = Path("results")


def load(pattern):
    found = sorted(results.glob(pattern))
    return json.loads(found[0].read_text(encoding="utf-8")) if found else None


adapt = load("adapt_r2.100.json")
hivqe = load("hivqe_r2.100_sector.json")
classical = load("classical*.json")

if adapt is None:
    raise SystemExit("No ADAPT-VQE result yet -- run section 5.")

reference = adapt["reference_energy"]
hartree_fock = adapt["hartree_fock_energy"]
correlation = reference - hartree_fock

rows = []


def add(kind, method, energy, cost):
    if energy is None:
        return
    error = (energy - reference) * HARTREE_TO_MHA
    recovered = (energy - hartree_fock) / correlation * 100 if correlation else float("nan")
    rows.append((kind, method, energy, error, recovered, cost))


add("classical", "RHF (mean field)", hartree_fock, "seconds")

# classical.json is keyed by distance; find the 2.10 entry whatever its exact key.
entry = None
if classical:
    for key, value in classical.items():
        if isinstance(value, dict) and abs(float(key) - 2.10) < 1e-6:
            entry = value
            break
if entry:
    add("classical", "MP2", entry.get("mp2_total"), "seconds")
    add("classical", "CCSD", entry.get("ccsd_total"), "seconds")
    add("classical", "CCSD(T)", entry.get("ccsd_t_total"), "seconds")

add("quantum", f"ADAPT-VQE ({adapt['n_parameters']} operators)", adapt["energy"],
    f"{adapt['energy_evaluations']:,} energy evals, {adapt['seconds']/60:.0f} min")
if hivqe:
    add("quantum", f"HI-VQE ({hivqe['dimension']:,} dets)", hivqe["energy"],
        f"{hivqe['iterations']} iterations, {hivqe.get('seconds', 0)/60:.1f} min")

add("exact", f"CASCI ({adapt['full_cas_determinants']:,} dets)", reference, "the yardstick")

width = max(len(row[1]) for row in rows)
print(f"Li2S  r = 2.100 A   CAS(12e,12o)   24 qubits   "
      f"{adapt['full_cas_determinants']:,} determinants")
print()
print(f"{'':<9} {'method':<{width}} {'energy (Ha)':>16} {'err (mHa)':>11} "
      f"{'corr %':>8}  cost")
print("-" * (9 + width + 16 + 11 + 8 + 30))
for kind, method, energy, error, recovered, cost in rows:
    flag = ""
    if kind != "exact":
        flag = " *" if abs(error) <= 1.6 else ""
    print(f"{kind:<9} {method:<{width}} {energy:>16.9f} {error:>11.4f} "
          f"{recovered:>8.2f}  {cost}{flag}")
print()
print("  * inside chemical accuracy (1.6 mHa)")


## 8. The figure

Two panels. Left: error against CASCI, log scale, with the 1.6 mHa line — classical
methods in the warm colours, quantum in blue. Right: how ADAPT-VQE actually got there,
error against ansatz length, which is the plot that shows whether it *converged* or
merely *stopped*.

A flattening curve on the right means the pool is exhausted and you are looking at the
ansatz ceiling. A curve still descending at the last point means it ran out of
operator budget, and the number is not yet the method's best.

In [ ]:
%matplotlib inline
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

results = Path("results")
adapt = json.loads((results / "adapt_r2.100.json").read_text(encoding="utf-8"))
reference = adapt["reference_energy"]

COLOURS = {"RHF": "#8b949e", "MP2": "#d29922", "CCSD": "#db6d28",
           "CCSD(T)": "#a371f7", "ADAPT-VQE": "#1f6feb", "HI-VQE": "#0f8b3c"}

bars = [("RHF", adapt["hartree_fock_energy"])]
found = sorted(results.glob("classical*.json"))
if found:
    payload = json.loads(found[0].read_text(encoding="utf-8"))
    for key, value in payload.items():
        if isinstance(value, dict) and abs(float(key) - 2.10) < 1e-6:
            for label, field in (("MP2", "mp2_total"), ("CCSD", "ccsd_total"),
                                 ("CCSD(T)", "ccsd_t_total")):
                if value.get(field) is not None:
                    bars.append((label, value[field]))
bars.append(("ADAPT-VQE", adapt["energy"]))
hivqe_files = sorted(results.glob("hivqe_r2.100_sector.json"))
if hivqe_files:
    bars.append(("HI-VQE",
                 json.loads(hivqe_files[0].read_text(encoding="utf-8"))["energy"]))

figure, axes = plt.subplots(1, 2, figsize=(13, 5))

labels = [name for name, _ in bars]
errors = [max(abs(energy - reference) * 1000, 1e-4) for _, energy in bars]
axes[0].bar(labels, errors, color=[COLOURS[name] for name in labels])
axes[0].axhline(1.6, color="#2da44e", linestyle="--", linewidth=1.2,
                label="chemical accuracy (1.6 mHa)")
axes[0].set_yscale("log")
axes[0].set_ylabel("|E - E(CASCI)|  (mHa)")
axes[0].set_title("Error against the exact active-space energy", fontweight="bold")
axes[0].tick_params(axis="x", rotation=20)
axes[0].legend(frameon=False, fontsize=9)
axes[0].grid(axis="y", alpha=0.25)

steps = [record["n_parameters"] for record in adapt["iterations"]]
trace = [max(abs(record["error_millihartree"]), 1e-4) for record in adapt["iterations"]]
axes[1].plot(steps, trace, marker="o", markersize=4, color=COLOURS["ADAPT-VQE"],
             label="ADAPT-VQE")
axes[1].axhline(1.6, color="#2da44e", linestyle="--", linewidth=1.2,
                label="chemical accuracy (1.6 mHa)")
axes[1].set_yscale("log")
axes[1].set_xlabel("operators in the ansatz")
axes[1].set_ylabel("|E - E(CASCI)|  (mHa)")
axes[1].set_title("Did ADAPT-VQE converge, or just stop?", fontweight="bold")
axes[1].legend(frameon=False, fontsize=9)
axes[1].grid(alpha=0.25)

figure.suptitle(
    f"Li$_2$S, 24 qubits, r = 2.10 A, {adapt['full_cas_determinants']:,} determinants",
    fontweight="bold",
)
figure.tight_layout()
Path("results/figures").mkdir(parents=True, exist_ok=True)
figure.savefig("results/figures/adapt_comparison.png", dpi=150, bbox_inches="tight")
print("saved results/figures/adapt_comparison.png")
plt.show()


## 9. How to read this

**If ADAPT-VQE reached chemical accuracy:** it did so with exact expectation values and
no measurement cost. On real hardware it would still owe 15,697 Pauli-word measurement
settings *per optimizer step*, times its energy-evaluation count. Multiply those two
numbers before drawing any conclusion about practicality — that product is the honest
cost, and it is the reason the field moved on from VQE.

**If it did not:** the pool had nothing left worth adding, and the gap is the ansatz
ceiling. No amount of better hardware, more shots or noise mitigation closes it.

**Either way, note what HI-VQE spent** to reach its own number: a few minutes, one
measurement setting per iteration, and ~2% of the determinant space.

**And the standing caveat.** At 24 qubits CASCI solves this exactly. Nothing here is
evidence of quantum advantage — it is a controlled comparison at a size where the true
answer is known, which is the only place a comparison like this can be trusted.

## 10. Download

In [ ]:
import shutil
from pathlib import Path

for name in ("results", "cache"):
    if Path(name).is_dir():
        archive = shutil.make_archive(f"/content/adapt_{name}", "zip", ".", name)
        print(f"{archive}  ({Path(archive).stat().st_size / 1e6:.2f} MB)")

try:
    from google.colab import files
    for name in ("results", "cache"):
        path = Path(f"/content/adapt_{name}.zip")
        if path.is_file():
            files.download(str(path))
except ImportError:
    print("Not on Colab -- the zips are next to this notebook.")
